In [ ]:
import cv2
from ultralytics import YOLO
import random
import time
CLASS_COLORS = {
    0: (0, 255, 0),      # Зеленый — например, "Свободно" / "Чисто"
    1: (0, 165, 255),    # Оранжевый — "В процессе"
    2: (0, 0, 255),      # Красный — "Занято" / "Проблема"
    3: (255, 0, 0),      # Синий — другой статус
    # Добавляйте свои классы по мере необходимости
}

# Названия классов (для отображения текста)
CLASS_NAMES = {
    0: "В работе",
    1: "свободно",
    2: "беспорядок",
    3: "отсутствие активности",
}
def get_color(cls_id):
    """Возвращает цвет по классу, если класса нет — случайный"""
    return CLASS_COLORS.get(int(cls_id), (255, 255, 255))  # белый по умолчанию
def process_video_with_tracking(model, input_video_path, show_video=True, save_video=False,output_video_path="output_video.mp4"):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
       raise Exception("Error: Could not open video file.")
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = model.track(frame, iou=0.4, conf=0.5, persist=True, imgsz=480, verbose=False, tracker="bytetrack.yaml")

        if results[0].boxes.id != None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = box
                cls_id = cls_ids[i]
                color = get_color(cls_id)

                # Рисуем прямоугольник цветом класса
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

                 # Подписываем класс + ID
                label = f"{CLASS_NAMES.get(cls_id, f'cls{cls_id}')} ID:{ids[i] if ids is not None else ''}"

                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        if save_video:
            out.write(frame)
        if show_video:
            frame = cv2.resize(frame, (0,0), fx=2, fy=1.4)
            cv2.imshow("frame", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    if save_video:
        out.release()
cv2.destroyAllWindows()
model = YOLO('runs/detect/train/weights/best.pt')
model.fuse()
process_video_with_tracking(model, 0, show_video=True, save_video=False, output_video_path="output_video.mp4")



Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
